# Python Generics

[Material Used](https://docs.python.org/3/library/typing.html)

Generics are nothing more than type hints in Python. The package that holds functionality for type hints is called `typing`. There is also a new package that holds functionality of collection type hints called `collections.abc`. 

Generics:
* Allow functions, methods and classes to work with arguments of any type
* Better define how types can mix
* Tools such as `mypy` can be used to catch type mismatches during static analysis of the code

**Note:** Python runtime does not enforce function and variable type annotations. Such annotations are used by third party tools.

**Note:** Have `nb_mypy` package installed in your conda environment where this notebook is running to produce all of the below type checking errors.

In [1]:
# load mypy to print error messages when type mismatch happens
%load_ext nb_mypy

Version 1.0.4


In [2]:
def greetings(name: str) -> str:
    return f"Hello {name}"

In [3]:
print(greetings("Tigran"), " -- ", greetings(5))

<cell>1: error: Argument 1 to "greetings" has incompatible type "int"; expected "str"


Hello Tigran  --  Hello 5


## Type Aliases

In [4]:
Vector = list[float]


def scale(scalar: float, vector: Vector) -> Vector:
    return [scalar * num for num in vector]


new_vector = scale(2.0, [1.0, -4.2, 5.4])

In [5]:
type(new_vector), new_vector

(list, [2.0, -8.4, 10.8])

Use type aliases to simplify complex type signatures, e.g. 
```
from collections.abc import Sequence

ConnectionOptions = dict[str, str]
Address = tuple[str, int]
Server = tuple[Address, ConnectionOptions]

def broadcast(message: str, servers: Sequence[Server])
    ...
```

## NewType from `typing` Module

In [92]:
from typing import NewType

# The form is: Derived = NewType('Derived', Base)
UserId = NewType("UserId", int)

some_id = UserId(111)
print(type(some_id), some_id)

<class 'int'> 111


In [93]:
# does not create a new class though
class AdminUserId(UserId):
    pass

<cell>2: error: Cannot subclass "NewType"


TypeError: NewType.__init__() takes 3 positional arguments but 4 were given

In [10]:
# Derived type name and the name passed to NewType must match
BlahStr = NewType("BlahStr_Other", str)

<cell>2: error: String argument 1 "BlahStr_Other" to NewType(...) does not match variable name "BlahStr"


In [11]:
output = UserId(1111) + UserId(222)
output

1333

In [12]:
def get_user_name(user_id: UserId) -> str:
    return str(user_id)

In [13]:
user_a = get_user_name(UserId(111))  # passes
user_b = get_user_name(111)  # fails

<cell>2: error: Argument 1 to "get_user_name" has incompatible type "int"; expected "UserId"


## Callable from `collections.abc`

Use to type-hint callback functions using `Callable[[Arg1Type, Arg2Type], ReturnType]` form. 

In [14]:
from collections.abc import Callable


def feeder(get_next_item: Callable[[], str]) -> None:
    print(get_next_item())

In [15]:
feeder(lambda: "Blah")  # passes
feeder(lambda: 5)  # fails

<cell>2: error: Argument 1 to "feeder" has incompatible type "Callable[[], int]"; expected "Callable[[], str]"
<cell>2: error: Incompatible return value type (got "int", expected "str")


Blah
5


In [16]:
from typing import Awaitable


def async_query(
    on_success: Callable[[int], None], on_error: Callable[[int, Exception], None]
) -> None:
    pass


async def on_update(value: str) -> None:
    pass


callback: Callable[[str], Awaitable[None]] = on_update

In [17]:
callback2: Callable[[str], float] = on_update

<cell>1: error: Incompatible types in assignment (expression has type "Callable[[str], Coroutine[Any, Any, None]]", variable has type "Callable[[str], float]")


## Map and Collection Type Hints

Type information about collection objects cannot be statically inferred in Python. So, `Mapping` and `Sequence` (base abstract?) classes are offered to denote expected types for maps and collections.

In [18]:
from collections.abc import Mapping, Sequence

In [19]:
class Employee:
    def __init__(self, n):
        self.name = n

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, n):
        self._name = n


def notify(employees: Sequence[Employee], overrides: Mapping[str, str]) -> None:
    for e in employees:
        print(e.name)

In [20]:
employees = [Employee("John"), Employee("Jane")]
notify(employees, dict({"blah": "b"}))  # passes

John
Jane


In [21]:
class Person:
    def __init__(self, n):
        self.name = n

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, n):
        self._name = n


notify([Person("John"), Person("Jane")], dict())  # fails

<cell>14: error: List item 0 has incompatible type "Person"; expected "Employee"
<cell>14: error: List item 1 has incompatible type "Person"; expected "Employee"


John
Jane


In [22]:
notify(employees, dict({"blah": 2}))

<cell>1: error: Dict entry 0 has incompatible type "str": "int"; expected "str": "str"


John
Jane


## Generics

### Functions

In [23]:
from typing import TypeVar
from collections.abc import Sequence

In [24]:
T = TypeVar("T")

# This is a generic function
def first(l: Sequence[T]) -> T:
    return l[0]

In [25]:
first([1, 2, 3]), first(["one", "two", "three"]), first(["mixed one", 2, None])

(1, 'one', 'mixed one')

In [26]:
T

~T

### Classes

In [27]:
from typing import Generic
from collections.abc import Iterable


# Takes single type parameter, see Generic[T]
class MyEntry(Generic[T]):
    def __init__(self, name: str, value: T) -> None:
        self._name = name
        self._value = value
    
    @property
    def name(self) -> str:
        return self._name
    
    @name.setter
    def name(self, n: str) -> None:
        self._name = n
    
    @property
    def value(self) -> T:
        return self._value
    
    @value.setter
    def value(self, v: T) -> None:
        self._value = v
        
    def __str__(self):
        return f"{self.name}:{str(self.value)}"

In [28]:
entry1 = MyEntry("John", "blah")
entry2 = MyEntry("Jane", 5)

print(entry1, entry2)

John:blah Jane:5


In [29]:
def entries_setter(entries: Iterable[MyEntry[str]]) -> None:
    for entry in entries:
        entry.name = "Tigran"

entries = [MyEntry("John", "blah"), MyEntry("Jane", "blah2")]
entries_setter(entries)

for entry in entries:
    print(entry.name, ":", entry.value)

Tigran : blah
Tigran : blah2


In [30]:
entries = [MyEntry("John", 4), MyEntry("Jane", 5)]
entries_setter(entries)

for entry in entries:
    print(entry.name, ":", entry.value)

<cell>2: error: Argument 1 to "entries_setter" has incompatible type "List[MyEntry[int]]"; expected "Iterable[MyEntry[str]]"


Tigran : 4
Tigran : 5


#### Bound the Generic Types

`TypeVar` types can be of 3 types:
* invariant
* covariant
* contravariant

Let's say we have `Base` class and `Derived` class. We know from OOP that we can use `Derived` where `Base` is expected, and we can use `Base` where `Derived` is expected (kind of). But, does that mean `List<Base>` can be used anywhere `List<Derived>` is expected, or vice versa? 

* `List` is `covariant` if `List<Base>` can be used where `List<Derived>` is expected -- I think this is equivalent to Java's `super` relationship and calling them `Base` and `Derived` here is not relevant
* `List` is `contravariant` if `List<Derived>` can be used where `List<Base>` is expected
* `List` is `invariant` if no parent-chald relationship exists between `List<Base>` and `List<Derived>`. This is the default behavior.

`TypeVar`'s `bound` attribute defines upper bound to the type, so the type value can only be the `bound` or a type that is derived from it.

In [31]:
from typing import TypeVar, Generic, Sequence
from collections.abc import Iterable, Iterator

T = TypeVar("T", contravariant=True)
B = TypeVar("B", bound=Iterable[str], covariant=True)
S = TypeVar("S", int, str) # int or string (or their child classes) would do

In [32]:
# Say, T represents some kind of List type that is iterable (see __iter__ method)
class MyList(Generic[T]):
    def __init__(self, items: Iterable[T]) -> None:
        self._my_list = items
        self._my_iter = iter(items)
        
    def __iter__(self) -> Iterator[T]:
        return self._my_iter             

class Base(str): pass
class Derived(Base): pass


#### Test Contravariance

In [33]:
def list_items(items: MyList[Derived]) -> None:
    for item in items:
        print(item)

# We can pass MyList[Base] where MyList[Derived] is expected due to contravariance
items = MyList([Base("blah"), Base("blah2")])
list_items(items)


blah
blah2


In [34]:
def list_items(items: MyList[Base]) -> None:
    for item in items:
        print(item)

# We cannot pass MyList[Derived] where MyList[Base] is expected due to covariance being False for T
items = MyList([Derived("blah"), Derived("blah2")])
list_items(items)

<cell>7: error: Argument 1 to "list_items" has incompatible type "MyList[Derived]"; expected "MyList[Base]"


blah
blah2


#### Test Covariance

In [35]:
# Say, B represents some kind of list type that is iterable (see __iter__ method)
class MyOtherList(Generic[B]):
    def __init__(self, items: Iterable[B]) -> None:
        self._my_list = items
        self._my_iter = iter(items)
        
    def __iter__(self) -> Iterator[B]:
        return self._my_iter  

In [36]:
def list_items(items: MyOtherList[Derived]) -> None:
    for item in items:
        print(item)

# We cannot pass MyList[Base] where MyList[Derived] is expected due to contravariance being False for B
items = MyOtherList([Base("blah"), Base("blah2")])
list_items(items)

<cell>7: error: Argument 1 to "list_items" has incompatible type "MyOtherList[Base]"; expected "MyOtherList[Derived]"


blah
blah2


In [37]:
def list_items(items: MyOtherList[Base]) -> None:
    for item in items:
        print(item)

# We can pass MyList[Derived] where MyList[Base] is expected due to covariance
items = MyOtherList([Derived("blah"), Derived("blah2")])
list_items(items)

blah
blah2


#### Test Invariance

In [38]:
L = TypeVar("L")

# Say L represents some kind of list type that is iterable (see __iter__ method)
class YetAnotherList(Generic[L]):
    def __init__(self, items: Iterable[L]) -> None:
        self._my_list = items
        self._my_iter = iter(items)
        
    def __iter__(self) -> Iterator[L]:
        return self._my_iter

In [39]:
def list_items(items: YetAnotherList[Base]) -> None:
    for item in items:
        print(item)
        
# Even though Derived is a child of Base, due to invariance YetAnotherList[Derived] cannot be used where YetAnotherList[Base] is expected
items = YetAnotherList([Derived("blah"), Derived("blah2")])
list_items(items)

<cell>7: error: Argument 1 to "list_items" has incompatible type "YetAnotherList[Derived]"; expected "YetAnotherList[Base]"


blah
blah2


In [40]:
def list_items(items: YetAnotherList[Derived]) -> None:
    for item in items:
        print(item)
        
# Even though Base is a parent of Derived, due to invariance YetAnotherList[Base] cannot be used where YetAnotherList[Derived] is expected
items = YetAnotherList([Base("blah"), Base("blah2")])
list_items(items)

<cell>7: error: Argument 1 to "list_items" has incompatible type "YetAnotherList[Base]"; expected "YetAnotherList[Derived]"


blah
blah2


### Multiple Types

In [41]:
# Can't use the same type twice
class Pair(Generic[T,T]): pass

<cell>2: error: Duplicate type variables in Generic[...] or Protocol[...]


TypeError: Parameters to Generic[...] must all be unique

In [43]:
class Weird(Generic[T,B,S]): pass

### Multiple Inheritance with Generics

In [44]:
from collections.abc import Sized

class LinkedList(Sized, Generic[T]): pass 
class MyDict(Mapping[str, T]): pass # Some type variables could be fixed

### Generic Type Aliases

In [45]:
from collections.abc import Iterable

S = TypeVar("S")
Response = Iterable[S] | int

def response(query: str) -> Response[str]:
    return f"queried {query}"

def response2(query: str) -> Response:
    return 10

response("blah"), response2("blah")

('queried blah', 10)

In [46]:
T = TypeVar("T", int, float, complex) # T can be used for int, float and complex numbers
Vec = Iterable[tuple[T, T]] # alias

def in_product(v: Vec[T]) -> T:
    return sum(x*y for x,y in v)

In [47]:
in_product([(1, 2), (2.2, 3.3), (complex(1,2), 4.4)])

(13.66+8.8j)

In [48]:
class MyFloat(float): pass

# will work because MyFloat extends float
in_product([(1, 1.1), (MyFloat(2.2), complex(1,2))])

(3.3000000000000003+4.4j)

### ParamSpec and Concatenate

In [49]:
from typing import Generic, TypeVar, ParamSpec

T = TypeVar("T")
P = ParamSpec("P")

class Z(Generic[T,P]): ...

`ParamSpec` is created to support decorators so that type hinting doesn't fail when decorators have to return various types.

In [50]:
from typing import Optional

def ignore_exception(function: Callable[P, T]) -> Callable[P, Optional[T]]:
    def decorator(*args: P.args, **kwargs: P.kwargs) -> Optional[T]:
        try:
            return function(*args, **kwargs)
        except Exception:
            return None
    return decorator


In [51]:
@ignore_exception
def divide(num: int, denom: int) -> float:
    return num / denom

divide(1,2), divide(1,0)

(0.5, None)

Now imagine `ParamSpec` did not exist. What would the `ignore_exception` decorator take and return when we want to type-proof it? Perhaps it should take `Callable` to represent any function that it aims to decorate, and return `Callable[[Any, Any], Optional[T]]` to capture the decorated function with its args captured via `Any` and return captured via `Optional[T]` (to represent the fact that the decorated function does not always return the target type). Perhaps `Any`s can even be replaced with `VarArg[Any]` and `KwArg[Any]` to be even more specific that they are expected to be the function arguments.

In [52]:
from typing import Any

C = TypeVar('C', bound=Callable)

def ignore_exception2(function: C) -> Callable[[Any, Any], Optional[T]]:
    def decorator(*args, **kwargs) -> Optional[T]:
        try:
            return function(*args, **kwargs)
        except Exception:
            return None
    return decorator

In [53]:
@ignore_exception2
def divide2(num: int, denom: int) -> float:
    return num / denom

divide2(1,2), divide2(1,0)

(0.5, None)

Because some decorators modify the argument types, there is a `Concatinate` that allows prepending, appending, transforming or removing function arguments.

In [54]:
from typing import ParamSpec, Concatenate

In [55]:
T = TypeVar("T")
P = ParamSpec("P")

#### Add Attribute

In [56]:
# Add a string attribute to a function to print before applying the function
def print_wrapper(function: Callable[P, T]) -> Callable[Concatenate[str, P], T]:
    def decorator(to_print: str, *args, **kwargs) -> T:
        print(to_print)
        return function(*args, **kwargs)
    return decorator

In [57]:
@print_wrapper
def add10(x: int) -> int:
    return x + 10

In [58]:
add10("print me", 10)

print me


20

#### Remove Attribute

In [59]:
def first_attribute_remover(function: Callable[Concatenate[int, P], T]) -> Callable[P, T]:
    def decorator(*args, **kwargs) -> T:
        return function(10000000000, *args, **kwargs)
    return decorator

In [60]:
@first_attribute_remover
def add10(unused: int, number: int) -> int:
    return number + 10

@first_attribute_remover
def add10_use_unused(unused: int, number: int) -> int:
    return unused + number + 10

# with and without unused being used
add10(10), add10_use_unused(10)

(20, 10000000020)

#### Transorm Attributes

In [61]:
# Changing the argument type from int to str
def first_argument_transformer(function: Callable[Concatenate[int, P], T]) -> Callable[Concatenate[str, P], T]:
    def decorator(*args, **kwargs) -> T:
        return function(*args, **kwargs)
    return decorator

In [62]:
@first_argument_transformer
def func(i: int) -> None:
    print(i)
    
func(1)

<cell>5: error: Argument 1 to "func" has incompatible type "int"; expected "str"


1


### Any Type

Static type checker will treat every type being compatible with `Any`. 
All functions without a specified return type or argument types will default to using `Any`.

In [63]:
from typing import Any

a: Any = None
a = []
a = 10

# Assign Any to any variable, passes type checking. No type checking is even performed.
s: str = ""
s = a 
print(s)

10


In [64]:
s = 10

<cell>1: error: Incompatible types in assignment (expression has type "int", variable has type "str")


Use `Any` as an escape hatch to mix dynamically and statically typed code.

When type checking is concerned, unlike `object`, `Any` is a supertype for every other type. And like `object`, `Any` is a subtype of every other type.

In [65]:
def magic_a(item: object) -> None:
    item.magic()

<cell>2: error: "object" has no attribute "magic"


In [66]:
def magic_b(item: Any) -> None:
    item.magic()

### Nominal vs Structural Subtyping

Initially Python's static type system was defined as `nominal subtyping`, meaning that class `A` is allowed everywhere where class `B` is expected if and only if `A` is a subclass of `B`. This behavior was also applied to abstract base classes (e.g. `Iterable`). 

In [67]:
from collections.abc import Sized, Iterable, Iterator
from typing import Union

class Bucket(Sized, Iterable[int]):
    def __init__(self, items: Union[Sized, Iterable[int]]):
        self._items = items
        
    def __len__(self) -> int:
        return len(self._items)
    
    def __iter__(self) -> Iterator[int]:
        return iter(self._items)
    
def sum_all(items: Iterable[int]) -> int:
    return sum(items)

sum_all(Bucket([1,2,3]))

<cell>9: error: Argument 1 to "len" has incompatible type "Union[Sized, Iterable[int]]"; expected "Sized"
<cell>12: error: Argument 1 to "iter" has incompatible type "Union[Sized, Iterable[int]]"; expected "SupportsIter[Iterator[int]]"


6

Type checking later started allowing code without explicity specifying base classes in the class definition:

In [68]:
class Bucket2: # No base classes
    def __init__(self, items: Union[Sized, Iterable[int]]):
        self._items = items
        
    def __len__(self) -> int:
        return len(self._items)
    
    def __iter__(self) -> Iterator[int]:
        return iter(self._items)

<cell>6: error: Argument 1 to "len" has incompatible type "Union[Sized, Iterable[int]]"; expected "Sized"
<cell>9: error: Argument 1 to "iter" has incompatible type "Union[Sized, Iterable[int]]"; expected "SupportsIter[Iterator[int]]"


In [69]:
sum_all(Bucket2([1, 2, 3]))

6

`Bucket2` is implicitly considered a subtype of `Iterable` and `Sized` by static type checkers and hence the above code passes type check. This is called `structural subtyping` or `duck-typing`. The types are inferred from `__iter__` and `__len__` respectively. 

Moreover, one can define custom protocol by subclassing a special `Protocol` class to fully leverage duck-typing:

In [70]:
from typing import Protocol

# Protocol requires covariant generic types
T = TypeVar("T", covariant=True)

# Protocol can be generic or not
class Proto(Protocol[T]):
    def custom_func(self) -> int:
        pass
    
    def another_func(self) -> T:
        pass
    
class C:
    def custom_func(self) -> int:
        return 0
    
    def another_func(self) -> int:
        return 1

def func(x: Proto) -> int:
    try:
        return x.custom_func()
    except Exception:
        return 0

func(C()) # passes type checking as long as C has the same custom proto format

0

In [71]:
class D:
    def another_func(self) -> int:
        return 0
    
func(D())

<cell>5: error: Argument 1 to "func" has incompatible type "D"; expected "Proto[Any]"
<cell>5: note: "D" is missing following "Proto" protocol member:
<cell>5: note:     custom_func


0

### Literals

Use `Literal` to enfoce predefined `string` values via type checking: 

In [72]:
from typing import Literal 

MODE = Literal["r", "rb", "w", "wb"]
def file_open_helper(file: str, mode: MODE):
    pass

file_open_helper("file1", "typo")

<cell>7: error: Argument 2 to "file_open_helper" has incompatible type "Literal['typo']"; expected "Literal['r', 'rb', 'w', 'wb']"


In [73]:
some_str: MODE = "rr"

<cell>1: error: Incompatible types in assignment (expression has type "Literal['rr']", variable has type "Literal['r', 'rb', 'w', 'wb']")


In [74]:
mode: Literal["r", "w"] = "w"

In [75]:
mode: Literal["r", "w"] = "www"

<cell>1: error: Incompatible types in assignment (expression has type "Literal['www']", variable has type "Literal['r', 'w']")


### Final

Enforces variable to be final (cannot be re-assigned) via type check:

In [76]:
from typing import Final

MAX_SIZE: Final = 10000

In [77]:
MAX_SIZE += 1

<cell>1: error: Cannot assign to final name "MAX_SIZE"


### ClassVar

In [78]:
from typing import ClassVar

class A:
    count: ClassVar[int] = 0

In [79]:
A.count += 1 # passes

In [80]:
a_obj = A()
a_obj.count += 1

<cell>2: error: Cannot assign to class variable "count" via instance


### Optional

`Optional[X]` is equivalent to `X | None`, or `Union[X, None]`. 
> Note, that this is not the same as optional argument.

In [81]:
from typing import Optional

def foo(arg: Optional[int] = None) -> None:
    if arg is not None:
        print(arg)

In [82]:
foo(2)

2


In [83]:
foo()

In [84]:
foo(None)

### NamedTuple

Typed version of `collections.namedtuple()` which defines a tuple that has a name:

In [85]:
from typing import NamedTuple
import collections

In [86]:
class Employee(NamedTuple):
    name: str
    id: int = 3
    
# Same as this
Employee2 = collections.namedtuple("Employee2", ["name", "id"])

In [87]:
type(Employee) == type(Employee2)

True

In [88]:
emp = Employee("Tigran") 
assert emp.id == 3

emp2 = Employee2("Tigran", 4)

In [89]:
emp

Employee(name='Tigran', id=3)

In [90]:
emp2

Employee2(name='Tigran', id=4)